# 🧱 Lab: Ingestion Lab — 600 Messy Rows into Clean Records

**Module 3: Real Data: Paths, CSV, JSON, pandas** | **Type: Wall Lab**

---

## Learning Objectives

By the end of this lab, you will be able to:

1. **Build** a portable file-loading pipeline using `pathlib.Path` and explicit encodings
2. **Clean** a 600-row messy dataset step-by-step while reporting how many rows each step removed
3. **Assert** row counts and dtypes on the cleaned output so downstream code will not get garbage
4. **Produce** validated JSONL records with a full audit trail documenting what was dropped and why

## Concepts Covered

| Concept | Gloss | From |
|---------|-------|------|
| Filesystem Paths | Path objects, not strings; relative paths depend on working directory | mini-path-safe-loader |
| Text Encoding | Bytes become text only under a declared encoding, usually UTF-8 | mini-path-safe-loader |
| Counted Cleaning | Every cleaning step reports how many rows it removed | mini-clean-messy-dataframe |
| Dataclass as a shape | Function for behaviour, dataclass for shape | mini-add-type-hints |
| Pure function | Inputs via parameters, results via return | mini-extract-pure-function |

## Prerequisites

- **mini-path-safe-loader**: Portable paths and encoding errors
- **mini-csv-json-jsonl**: CSV/JSON both ways and JSON Lines
- **mini-clean-messy-dataframe**: Counted pandas cleaning


## Setup

In [1]:
import csv
import json
import random
from pathlib import Path

import numpy as np
import pandas as pd
from IPython.display import display, Markdown

def md(text):
    """Display text as rendered markdown."""
    display(Markdown(text))

print("Imports ready.")

Imports ready.


## Step 1 — Generate the Messy Source File

In a real project the messy CSV arrives from an export or API dump. Here we'll generate 600 rows with realistic data-quality problems — the same kinds you saw in `mini-clean-messy-dataframe`, but at scale.

The problems seeded into the data:

| Problem | Example |
|---------|--------|
| Missing `text` | `None` or empty string |
| Whitespace-only `text` | `"   "` |
| Missing `role` | `None` |
| Invalid `role` | `"bot"`, `"human"` |
| Missing `id` | `None` |
| Duplicate rows | Exact copies |
| Negative token counts | `-3` |
| Leading/trailing whitespace in text | `"  How does RAG work?  "` |

In [2]:
# ---------- generate 600 messy rows ----------
random.seed(42)
np.random.seed(42)

good_texts = [
    "What is RAG?", "Explain embeddings.", "How does chunking work?",
    "What is fine-tuning?", "Tell me about vector databases.",
    "How do transformers work?", "What is tokenization?",
    "Explain attention mechanisms.", "What are LLM agents?",
    "Describe prompt engineering.", "What is cosine similarity?",
    "How do I evaluate a RAG pipeline?", "What is few-shot prompting?",
    "Explain chain-of-thought reasoning.", "What is JSON Lines?",
    "Retrieval-Augmented Generation combines search with LLMs.",
    "Embeddings are dense vector representations of text.",
    "Chunking splits documents into smaller pieces.",
    "Fine-tuning adapts a pre-trained model to a specific task.",
    "Vector databases store and query high-dimensional vectors.",
]

valid_roles = ["user", "assistant", "system"]
invalid_roles = ["bot", "human", "ai", "chatbot"]

rows = []
msg_id = 1

for i in range(600):
    r = random.random()

    # ~5% missing id
    rid = None if r < 0.05 else msg_id
    msg_id += 1

    # ~5% missing role, ~5% invalid role
    if r < 0.10:
        role = None
    elif r < 0.15:
        role = random.choice(invalid_roles)
    else:
        role = random.choice(valid_roles)

    # ~5% missing text, ~3% whitespace-only, ~4% empty string, ~8% extra whitespace
    if r < 0.05:
        text = None
    elif r < 0.08:
        text = "   "
    elif r < 0.12:
        text = ""
    elif r < 0.20:
        text = f"  {random.choice(good_texts)}  "
    else:
        text = random.choice(good_texts)

    # ~4% negative token count
    tokens = random.randint(-3, 0) if r < 0.04 else random.randint(2, 15)

    rows.append({"id": rid, "role": role, "text": text, "tokens": tokens})

# Inject ~30 exact duplicates by copying random existing rows
for _ in range(30):
    rows.append(dict(random.choice(rows[:570])))

random.shuffle(rows)
print(f"Generated {len(rows)} messy rows")

Generated 630 messy rows


## Step 2 — Write the Messy CSV (Portable Paths + Encoding)

We use `pathlib.Path` and explicit `encoding="utf-8"` — recall from `mini-path-safe-loader` that this ensures the file works identically on macOS, Windows, and Linux.

In [3]:
data_dir = Path("data")
data_dir.mkdir(exist_ok=True)

raw_csv_path = data_dir / "messy_messages.csv"

fieldnames = ["id", "role", "text", "tokens"]
with raw_csv_path.open("w", newline="", encoding="utf-8") as f:
    writer = csv.DictWriter(f, fieldnames=fieldnames)
    writer.writeheader()
    writer.writerows(rows)

print(f"Wrote {raw_csv_path}  ({raw_csv_path.stat().st_size:,} bytes)")
print(f"Path type: {type(raw_csv_path)}  ← pathlib, not a string")

Wrote data\messy_messages.csv  (27,716 bytes)
Path type: <class 'pathlib.WindowsPath'>  ← pathlib, not a string


## Step 3 — Load and Inspect

Read the CSV back with pandas (recall from `mini-csv-json-jsonl`: pandas auto-infers types, which is handy for analysis). Always pass `encoding="utf-8"` explicitly.

In [4]:
df = pd.read_csv(raw_csv_path, encoding="utf-8")
print(f"Loaded {len(df)} rows, {len(df.columns)} columns")
print(f"Columns: {list(df.columns)}")
print(f"\nDtypes:\n{df.dtypes}")
print(f"\nNull counts:\n{df.isnull().sum()}")
df.head(10)

Loaded 630 rows, 4 columns
Columns: ['id', 'role', 'text', 'tokens']

Dtypes:
id        float64
role       object
text       object
tokens      int64
dtype: object

Null counts:
id        22
role      53
text      48
tokens     0
dtype: int64


,id,role,text,tokens
0,99.0,assistant,Explain embeddings.,2
1,211.0,user,Describe prompt engineering.,12
2,300.0,user,Describe prompt engineering.,7
3,594.0,system,Chunking splits documents into smaller pieces.,9
4,447.0,user,Fine-tuning adapts a pre-trained model to a ...,12
5,347.0,user,What is tokenization?,12
6,1.0,user,What are LLM agents?,5
7,333.0,system,What is cosine similarity?,11
8,370.0,system,Vector databases store and query high-dimensio...,8
9,546.0,system,How do I evaluate a RAG pipeline?,4


## Step 4 — Counted-Cleaning Pipeline

Recall the pattern from `mini-clean-messy-dataframe`: after **every** filter step, print how many rows were removed. We'll reuse the same `cleaned()` helper and add the cleaning rules needed for this dataset.

In [5]:
def cleaned(df_before: pd.DataFrame, df_after: pd.DataFrame, step: str) -> pd.DataFrame:
    """Print how many rows a cleaning step removed, then return the cleaned DataFrame."""
    dropped = len(df_before) - len(df_after)
    print(f"  Step: {step:<40s} | dropped {dropped:>3d} | remaining {len(df_after):>3d}")
    return df_after

In [6]:
print(f"Starting rows: {len(df)}\n")

# 1. Drop rows where 'id' is missing
df_clean = cleaned(df, df.dropna(subset=["id"]), "drop missing id")

# 2. Drop rows where 'role' is missing
df_clean = cleaned(df_clean, df_clean.dropna(subset=["role"]), "drop missing role")

# 3. Drop rows where 'text' is missing, empty, or whitespace-only
df_clean["text"] = df_clean["text"].fillna("").str.strip()
mask_has_text = df_clean["text"].str.len() > 0
df_clean = cleaned(df_clean, df_clean[mask_has_text].copy(), "drop empty/whitespace text")

# 4. Drop exact duplicate rows
df_clean = cleaned(df_clean, df_clean.drop_duplicates(), "drop exact duplicates")

# 5. Keep only valid roles
allowed_roles = {"user", "assistant", "system"}
mask_valid_role = df_clean["role"].isin(allowed_roles)
df_clean = cleaned(df_clean, df_clean[mask_valid_role].copy(), "drop invalid roles")

# 6. Drop rows with non-positive token counts
mask_pos_tokens = df_clean["tokens"] > 0
df_clean = cleaned(df_clean, df_clean[mask_pos_tokens].copy(), "drop non-positive token counts")

# Ensure id is integer (pandas may have read it as float due to NaN)
df_clean["id"] = df_clean["id"].astype(int)
df_clean = df_clean.reset_index(drop=True)

print(f"\nFinal rows: {len(df_clean)}  (started with {len(df)})")

Starting rows: 630

  Step: drop missing id                          | dropped  22 | remaining 608
  Step: drop missing role                        | dropped  31 | remaining 577
  Step: drop empty/whitespace text               | dropped  13 | remaining 564
  Step: drop exact duplicates                    | dropped  28 | remaining 536
  Step: drop invalid roles                       | dropped  17 | remaining 519
  Step: drop non-positive token counts           | dropped   0 | remaining 519

Final rows: 519  (started with 630)


## Step 5 — Audit Report

A production pipeline should always produce an audit trail alongside the cleaned data. Let's build one.

In [7]:
total_dropped = len(df) - len(df_clean)

audit = {
    "source_file": str(raw_csv_path),
    "raw_rows": int(len(df)),
    "clean_rows": int(len(df_clean)),
    "total_dropped": int(total_dropped),
    "drop_rate": round(total_dropped / len(df), 4),
    "cleaning_steps": [
        "drop missing id",
        "drop missing role",
        "drop empty/whitespace text",
        "drop exact duplicates",
        "drop invalid roles (kept: user, assistant, system)",
        "drop non-positive token counts",
    ],
}

report_lines = [
    "### Ingestion Audit Report",
    "",
    "| Metric | Value |",
    "|--------|------:|",
    f"| Source file | `{audit['source_file']}` |",
    f"| Raw rows | {audit['raw_rows']} |",
    f"| Clean rows | {audit['clean_rows']} |",
    f"| Total dropped | {audit['total_dropped']} |",
    f"| Drop rate | {audit['drop_rate']:.1%} |",
]

md("\n".join(report_lines))

### Ingestion Audit Report

| Metric | Value |
|--------|------:|
| Source file | `data\messy_messages.csv` |
| Raw rows | 630 |
| Clean rows | 519 |
| Total dropped | 111 |
| Drop rate | 17.6% |

## Step 6 — Assert counts and dtypes on the clean data

Before writing the final output, check the cleaned table still has the shape you think it has. This is a **record-count and dtype assertion**, not a NumPy broadcasting lesson: if cleaning silently emptied a column or coerced integers to floats, you want to know now.


In [8]:
n_clean = len(df_clean)

assert n_clean > 0, "Cleaning dropped every row — stop and inspect CLEANING.md"
assert df_clean["tokens"].notna().all(), "token counts went missing after cleaning"
assert str(df_clean["tokens"].dtype).startswith("int"), (
    f"Expected integer token counts, got {df_clean['tokens'].dtype}"
)
assert (df_clean["tokens"] > 0).all(), "Found non-positive token counts after cleaning!"

print(f"clean rows: {n_clean}")
print(f"tokens dtype: {df_clean['tokens'].dtype}")
print(f"tokens min/max: {int(df_clean['tokens'].min())} / {int(df_clean['tokens'].max())}")


✅ token_arr shape: (519,)
✅ token_arr dtype: int64
✅ All values positive. Min=2, Max=15, Mean=8.5


## Step 7 — Write Clean Records as JSONL

Recall from `mini-csv-json-jsonl`: JSON Lines is appendable, streamable, and the standard format for LLM datasets. We'll write the cleaned data and the audit report to separate files.

In [9]:
# Write clean records as JSONL
clean_jsonl_path = data_dir / "clean_messages.jsonl"

records = df_clean.to_dict(orient="records")
with clean_jsonl_path.open("w", encoding="utf-8") as f:
    for rec in records:
        f.write(json.dumps(rec, ensure_ascii=False) + "\n")

print(f"Wrote {len(records)} clean records to {clean_jsonl_path}")
print(f"File size: {clean_jsonl_path.stat().st_size:,} bytes")

# Write audit report as JSON
audit_path = data_dir / "ingestion_audit.json"
audit_path.write_text(
    json.dumps(audit, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print(f"Wrote audit report to {audit_path}")

Wrote 519 clean records to data\clean_messages.jsonl


File size: 45,976 bytes
Wrote audit report to data\ingestion_audit.json


## Step 8 — Verify the Output End-to-End

A final sanity check: reload the JSONL we just wrote and confirm it matches the cleaned DataFrame. This closes the loop — we loaded messy data, cleaned it, wrote it out, and verified the output.

In [10]:
# Read the JSONL back
df_verify = pd.read_json(clean_jsonl_path, lines=True, encoding="utf-8")

# Assert shape matches
assert df_verify.shape == df_clean.shape, (
    f"Shape mismatch after reload: {df_verify.shape} vs {df_clean.shape}"
)

# Assert no nulls
assert df_verify.isnull().sum().sum() == 0, "Nulls found in reloaded data!"

# Assert all roles are valid
assert set(df_verify["role"].unique()).issubset(allowed_roles), "Invalid roles in output!"

# Assert all tokens positive
assert (df_verify["tokens"] > 0).all(), "Non-positive tokens in output!"

print(f"✅ Reloaded {len(df_verify)} rows from JSONL — all assertions passed")
print(f"✅ Roles present: {sorted(df_verify['role'].unique())}")
print(f"✅ Token range: {df_verify['tokens'].min()} – {df_verify['tokens'].max()}")
print(f"\nOutput files:")
for p in sorted(data_dir.iterdir()):
    print(f"  {p.name:30s}  {p.stat().st_size:>8,} bytes")

✅ Reloaded 519 rows from JSONL — all assertions passed
✅ Roles present: ['assistant', 'system', 'user']
✅ Token range: 2 – 15

Output files:
  clean_messages.jsonl              45,976 bytes
  ingestion_audit.json                 381 bytes
  messy_messages.csv                27,716 bytes


## 🎯 Summary

### What You Built

You built a complete ingestion pipeline that took 600 messy chat-message rows, cleaned them through six counted steps, validated the output with shape and dtype assertions, and wrote the results as portable JSONL — with a full audit report documenting exactly what was dropped and why.

### Key Takeaways

1. **Portable paths + explicit encoding** — using `pathlib.Path` and `encoding="utf-8"` ensures your pipeline runs identically on macOS, Windows, and Linux
2. **Counted cleaning** — printing row counts after every filter step makes data loss visible and auditable, not silent
3. **Shape and dtype assertions** — a few `assert` lines after cleaning catch silent bugs (wrong dimensions, wrong types, unexpected values) before they propagate
4. **Audit trail** — saving a JSON report alongside the cleaned data lets anyone reproduce or question the pipeline's decisions
5. **JSONL as the output format** — one record per line is appendable, streamable, and the standard for LLM datasets